In [1]:
import pandas as pd
from sqlalchemy import create_engine

username = "root"
password = "2003"
host = "localhost"
port = "3306"
database = "cart2insights"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

with engine.connect() as conn:
    print("Connected successfully!")

Connected successfully!


In [3]:
orders_features = pd.read_sql("""
    SELECT 
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,
        DATEDIFF(o.order_delivered_customer_date, o.order_purchase_timestamp) AS delivery_days,
        DATEDIFF(o.order_delivered_customer_date, o.order_estimated_delivery_date) AS delivery_delay,
        SUM(oi.price + oi.freight_value) AS total_order_value
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.order_id, o.customer_id, o.order_purchase_timestamp,
             o.order_delivered_customer_date, o.order_estimated_delivery_date
""", con=engine)

print(orders_features.shape)
orders_features.head()

(98666, 8)


,order_id,customer_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay,total_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,2017-09-13 08:59:02,2017-09-20 23:43:48,2017-09-29,7.0,-9.0,72.190001
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,2017-04-26 10:53:06,2017-05-12 16:04:24,2017-05-15,16.0,-3.0,259.829994
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,2018-01-14 14:33:31,2018-01-22 13:19:16,2018-02-05,8.0,-14.0,216.870001
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,2018-08-08 10:00:35,2018-08-14 13:32:39,2018-08-20,6.0,-6.0,25.780000
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,2017-02-04 13:57:51,2017-03-01 16:42:31,2017-03-17,25.0,-16.0,218.039993


In [10]:
customer_features = pd.read_sql("""
    SELECT 
        o.customer_id,
        COUNT(DISTINCT o.order_id) AS customer_order_count,
        SUM(oi.price + oi.freight_value) AS customer_total_spending,
        AVG(oi.price + oi.freight_value) AS customer_avg_order_value
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.customer_id
""", con=engine)

print(customer_features.shape)
customer_features.head()

(98666, 4)


,customer_id,customer_order_count,customer_total_spending,customer_avg_order_value
0,00012a2ce6f8dcda20d059ce98491703,1,114.740004,114.740004
1,000161a058600d5901f007fab4c27140,1,67.410002,67.410002
2,0001fd6190edaaf884bcaf3d49edf079,1,195.420006,195.420006
3,0002414f95344307404f0ace7a26f1d5,1,179.349995,179.349995
4,000379cdec625522490c315e70c7a9fb,1,107.010000,107.010000


In [5]:
seller_features = pd.read_sql("""
    SELECT 
        oi.seller_id,
        COUNT(DISTINCT oi.order_id) AS seller_order_count,
        SUM(oi.price + oi.freight_value) AS seller_revenue
    FROM order_items oi
    GROUP BY oi.seller_id
""", con=engine)

print(seller_features.shape)
seller_features.head()

(3095, 3)


,seller_id,seller_order_count,seller_revenue
0,0015a82c2db000af6aaaf3ae2ecb0532,3,2748.060001
1,001cca7ae9ae17fb1caed9dfb1094831,200,33934.170049
2,001e6ad469a905060d959994f1b41e4f,1,267.940001
3,002100f778ceb8431b7a1020ff7ab48f,51,2028.159999
4,003554e2dce176b5555353e4f3555ac8,1,139.379999


In [8]:
customer_features["repeat_customer"] = customer_features["customer_order_count"].apply(lambda x: 1 if x > 1 else 0)
customer_features.head()

,customer_id,customer_order_count,customer_total_spending,customer_avg_order_value,repeat_customer
0,00012a2ce6f8dcda20d059ce98491703,1,114.740004,114.740004,0
1,000161a058600d5901f007fab4c27140,1,67.410002,67.410002,0
2,0001fd6190edaaf884bcaf3d49edf079,1,195.420006,195.420006,0
3,0002414f95344307404f0ace7a26f1d5,1,179.349995,179.349995,0
4,000379cdec625522490c315e70c7a9fb,1,107.010000,107.010000,0


In [11]:
import os
os.makedirs("../data/features", exist_ok=True)

orders_features.to_csv("../data/features/orders_features.csv", index=False)
customer_features.to_csv("../data/features/customer_features.csv", index=False)
seller_features.to_csv("../data/features/seller_features.csv", index=False)

print("All feature tables saved.")

All feature tables saved.
